# CT-DGNN-JailGuard — Quickstart

This notebook shows the minimal end-to-end path: synthesize a tiny JailCampaign slice, build a graph, train one epoch, and print the Grönwall certificate.

In [ ]:
import sys
sys.path.insert(0, '..')

from pathlib import Path
from ct_dgnn.utils.seed import set_seed
set_seed(0)

In [ ]:
# 1. Synthesize a miniature JailCampaign (100 campaigns, 5k events).
!python ../scripts/build_jail_campaign.py \
    --out /tmp/jc_demo --n_campaigns 100 --total_events 5000

In [ ]:
# 2. Build the heterogeneous interaction graph.
from ct_dgnn.data.datasets import JailCampaignDataset
from ct_dgnn.data.graph_builder import APIInteractionGraphBuilder

ds = JailCampaignDataset('/tmp/jc_demo')
events = list(ds)[:256]
builder = APIInteractionGraphBuilder()
batch = builder.ingest(events)
print(batch.metadata)

In [ ]:
# 3. Instantiate the model, run a forward pass, inspect Lipschitz constants.
from ct_dgnn.models.ct_dgnn import CTDGNNJailGuard
from ct_dgnn.robustness.certificate import certificate_from_model

model = CTDGNNJailGuard(node_dims={'user': 64, 'session': 128, 'query': 384, 'model': 32})
out = model(batch, integrate_steps=1)
print('query logits:', out.query_logits.shape)
print('campaign logits:', out.campaign_logits.shape)
print('Lipschitz:', {k: float(v) for k, v in out.lipschitz.items()})

cert = certificate_from_model(model, k=50, T_seconds=86400, margin=1.0)
print('ε*:', cert.radius)